# Random Walk Skewness


In this problem we consider a <b>random walk</b> on the integers $\mathbb{Z}$, in which our position at time $t$ is denoted as $X_t$.


At time $0$ we start at position $0$. That is, $X_0=0$.<br>
At time $1$ we jump to position $1$. That is, $X_1=1$.<br>
Thereafter, at time $t=2,3,\dots$ we make a jump of size $|X_{t-2}|$ in either the positive or negative direction, with probability $1/2$ each way. If $X_{t-2}=0$ we stay put at time $t$.


At $t=5$ we find our position $X_5$ has the following distribution:
$$
X_5=\begin{cases}
-1\quad &\text{with probability }3/8\\
1\quad &\text{with probability }3/8\\
3\quad &\text{with probability }1/8\\
5\quad &\text{with probability }1/8\\
\end{cases}
$$

The <b>standard deviation</b> $\sigma$ of a <b>random variable</b> $X$ with <b>mean</b> $\mu$ is defined as

$$
\sigma=\sqrt{\mathbb{E}[X^2]-\mu^2}
$$
Furthermore the <b>skewness</b> of $X$ is defined as
$$
\text{Skew}(X)=\mathbb{E}\biggl[\Bigl(\frac{X-\mu}{\sigma}\Bigr)^3\biggr]
$$
For $X_5$, which has mean $1$ and standard deviation $2$, we find $\text{Skew}(X_5)=0.75$. You are also given $\text{Skew}(X_{10})\approx2.50997097$.

Find $\text{Skew}(X_{50})$. Give your answer rounded to eight digits after the decimal point.


## Solution.


In [1]:
from functools import cache
from collections import defaultdict
from pprint import pprint
from math import sqrt
from tqdm import tqdm

In [2]:
@cache
def X_dist(n):
    # returns
    # dist_n: {value: prob}
    # joint_n: {(x_n, x_{n-1}): prob}

    if n == 0:
        return {0: 1.0}, {}

    if n == 1:
        return {1: 1.0}, {(1, 0): 1.0}

    if n == 2:
        return {1: 1.0}, {(1, 1): 1.0}

    dist_prev, joint_prev = X_dist(n - 1)

    dist = defaultdict(float)
    joint = defaultdict(float)

    for (x_prev, x_prev2), p in joint_prev.items():
        step = abs(x_prev2)

        p_half = 0.5 * p

        x_plus = x_prev + step
        x_minus = x_prev - step

        dist[x_plus] += p_half
        dist[x_minus] += p_half

        joint[(x_plus, x_prev)] += p_half
        joint[(x_minus, x_prev)] += p_half

    return dict(dist), dict(joint)

In [ ]:
def mean(dist):
    return sum(x * px for x, px in dist.items())

def M2(dist):
    return sum(x**2 * px for x, px in dist.items())

def M3(dist):
    return sum(x**3 * px for x, px in dist.items())

def std(dist):
    return sqrt(M2(dist) - mean(dist)**2)

def skew(dist):
    mu = mean(dist)
    sigma = std(dist)
    return sum(((x-mu)/sigma)**3 * px for x, px in dist.items())

In [9]:
for n in range(21):
    print(n, len(X_dist(n)[0]), mean(X_dist(n)[0]), M2(X_dist(n)[0]), M3(X_dist(n)[0]))

0 1 0.0 0.0 0.0
1 1 1.0 1.0 1.0
2 1 1.0 1.0 1.0
3 2 1.0 2.0 4.0
4 3 1.0 3.0 7.0
5 4 1.0 5.0 19.0
6 5 1.0 8.0 40.0
7 7 1.0 13.0 97.0
8 10 1.0 21.0 217.0
9 14 1.0 34.0 508.0
10 19 1.0 55.0 1159.0
11 28 1.0 89.0 2683.0
12 38 1.0 144.0 6160.0
13 55 1.0 233.0 14209.0
14 76 1.0 377.0 32689.0
15 109 1.0 610.0 75316.0
16 156 1.0 987.0 173383.0
17 228 1.0 1597.0 399331.0
18 320 1.0 2584.0 919480.0
19 461 1.0 4181.0 2117473.0
20 665 1.0 6765.0 4875913.0


Easy to prove from the fact that $X_n=X_{n-1}+q\times | {X_{n-2}} |$ ($q$ is Binomial(1,-1)) that

- $E\left[X_n \right]=1$
- $E\left[X_n^2\right]=F_n$ (Fibonnaci)
- $E\left[X_n^3\right]= E\left[X_{n-1}^3\right] + 3 E\left[X_{n-1}X_{n-2}^2\right]=E\left[X_{n-1}^3\right] + 3 E\left[X_{n-2}^3\right]$
- $\text{skew} =1/\sigma^3 *\left(E\left[X_n^3\right] - 3\sigma^2 + 2\right)$.

In [18]:
@cache
def E1(n):
    return 1

@cache
def E2(n):
    if n == 1 or n == 2:
        return 1
    return E2(n-1) + E2(n-2)

@cache
def E3(n):
    if n == 1 or n == 2:
        return 1
    return E3(n-1) + 3*E3(n-2)


def skew(n):
    sigma = sqrt(E2(n) - E1(n)**2)
    return (1/sigma**3) * (E3(n) - 3 * E2(n) + 2)

In [24]:
round(skew(50), 8)

254.54470757